In [ ]:
"""
Fund Flow Prediction Model for KSE30 Index
Predicts future fund flows based on index weight changes and engineered features.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from xgboost import XGBRegressor, XGBClassifier


# =============================================================================
# Configuration
# =============================================================================
INPUT_FILE = "kse30_daily_data_engineered.xlsx"
OUTPUT_DIR = "fyp_outputs"

# Column names (after standardization)
DATE_COL = "DATE"
SYM_COL = "SYMBOL"
PRICE_COL = "PRICE"
VOLUME_COL = "VOLUME"
DELTA_WT_COL = "DELTA_WT"
DELTA_PRICE_COL = "DELTA_PRICE"
RETURN_COL = "RETURN"
DELTA_VOLUME_COL = "DELTA_VOLUME"
PREV_IDX_WT_COL = "PREV_IDX_WT"
PREV_FF_MCAP_COL = "PREV_FF_MCAP"
FLOW_PROXY_COL = "FLOW_PROXY"

# Feature columns
BASE_FEATURES = [
    "DELTA_WT",
    "DELTA_PRICE",
    "RETURN",
    "DELTA_VOLUME",
    "PREV_IDX_WT",
    "PREV_FF_MCAP",
]

ENGINEERED_FEATURES = [
    "RET_5",
    "RET_20",
    "VOLAT_5",
    "VOLAT_20",
    "VOL_SHOCK_5",
    "ABS_DELTA_WT",
    "DAYS_TO_REBAL",
]

ALL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

# Rebalancing dates (KSE30 typically rebalances twice yearly)
REBALANCING_DATES = [
    "2020-03-15", "2020-09-15",
    "2021-03-15", "2021-09-15",
    "2022-03-15", "2022-09-15",
    "2023-03-15", "2023-09-15",
    "2024-03-15", "2024-09-15",
    "2025-03-15", "2025-09-15",
]


# =============================================================================
# Data Loading & Preprocessing
# =============================================================================
def load_data(filepath: str) -> pd.DataFrame:
    """Load and clean the KSE30 data."""
    df = pd.read_excel(filepath)
    df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], unit='D', origin='1899-12-30')
    return df


def create_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create technical and momentum features."""
    df = df.copy()
    df = df.sort_values([SYM_COL, DATE_COL])

    # Price returns
    df["RET_5"] = df.groupby(SYM_COL)[PRICE_COL].pct_change(5)
    df["RET_20"] = df.groupby(SYM_COL)[PRICE_COL].pct_change(20)

    # Volatility
    df["VOLAT_5"] = df.groupby(SYM_COL)[PRICE_COL].rolling(5).std().reset_index(0, drop=True)
    df["VOLAT_20"] = df.groupby(SYM_COL)[PRICE_COL].rolling(20).std().reset_index(0, drop=True)

    # Volume shock
    df["VOL_SHOCK_5"] = df.groupby(SYM_COL)[VOLUME_COL].pct_change(5)

    # Absolute weight change
    df["ABS_DELTA_WT"] = df[DELTA_WT_COL].abs()

    # Days to next rebalancing
    rebal_timestamps = [pd.Timestamp(d) for d in REBALANCING_DATES]
    df["DAYS_TO_REBAL"] = df[DATE_COL].apply(
        lambda x: min([(d - x).days for d in rebal_timestamps if d > x])
        if any(d > x for d in rebal_timestamps) else np.nan
    )

    # Handle infinities
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df


def prepare_data(df: pd.DataFrame, features: list, target: str) -> pd.DataFrame:
    """Prepare data by dropping NaN values in features and target."""
    df = df.dropna(subset=[target] + features)
    return df


# =============================================================================
# Model Training
# =============================================================================
def train_regression_model(X_train, y_train):
    """Train XGBoost regressor for continuous flow prediction."""
    model = XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)
    return model


def train_classification_model(X_train, y_train):
    """Train XGBo classifier for Buy/Sell/Neutral prediction."""
    # Convert to labels: 1=Buy, 0=Neutral, -1=Sell -> 2, 1, 0 for XGBoost
    label_map = {-1: 0, 0: 1, 1: 2}
    reverse_map = {0: -1, 1: 0, 2: 1}

    y_mapped = y_train.map(label_map).values

    model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    )
    model.fit(X_train, y_mapped)
    return model, label_map, reverse_map


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_regression(y_true, y_pred):
    """Evaluate regression model performance."""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    print("=" * 50)
    print("REGRESSION MODEL EVALUATION")
    print("=" * 50)
    print(f"R² Score:  {r2:.4f}")
    print(f"MAE:       {mae:.6f}")
    print(f"MSE:       {mse:.8f}")
    print(f"RMSE:      {rmse:.6f}")
    print("=" * 50)

    return {"r2": r2, "mae": mae, "mse": mse, "rmse": rmse}


def evaluate_classification(y_true, y_pred, label_map):
    """Evaluate classification model performance."""
    # Map back to original labels
    reverse_map = {v: k for k, v in label_map.items()}
    y_true_orig = np.array([reverse_map.get(y, y) for y in y_true])
    y_pred_orig = np.array([reverse_map.get(y, y) for y in y_pred])

    acc = accuracy_score(y_true_orig, y_pred_orig)
    f1 = f1_score(y_true_orig, y_pred_orig, average="macro")

    print("=" * 50)
    print("CLASSIFICATION MODEL EVALUATION")
    print("=" * 50)
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score (macro): {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true_orig, y_pred_orig, labels=[-1, 0, 1]))
    print("=" * 50)

    # Confusion matrix
    cm = confusion_matrix(y_true_orig, y_pred_orig, labels=[-1, 0, 1])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Sell", "Neutral", "Buy"],
                yticklabels=["Sell", "Neutral", "Buy"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150)
    plt.close()

    return {"accuracy": acc, "f1_macro": f1, "confusion_matrix": cm}


def plot_feature_importance(model, feature_names, title="Feature Importance"):
    """Plot feature importance."""
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        return

    idx = np.argsort(importances)[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(idx)), importances[idx], align="center")
    plt.yticks(range(len(idx)), np.array(feature_names)[idx])
    plt.xlabel("Importance")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=150)
    plt.close()


def plot_predictions(y_true, y_pred, title="Predicted vs Actual"):
    """Plot predicted vs actual values."""
    plt.figure(figsize=(12, 5))
    plt.plot(y_true, label="Actual", alpha=0.7)
    plt.plot(y_pred, label="Predicted", alpha=0.7)
    plt.xlabel("Sample")
    plt.ylabel("Flow Proxy")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/predictions.png", dpi=150)
    plt.close()


# =============================================================================
# Main Pipeline
# =============================================================================
def main():
    print("=" * 60)
    print("KSE30 FUND FLOW PREDICTION")
    print("=" * 60)

    # 1. Load data
    print("\n[1] Loading data...")
    df = load_data(INPUT_FILE)
    print(f"    Loaded {len(df)} rows")

    # 2. Create engineered features
    print("\n[2] Creating engineered features...")
    df = create_engineered_features(df)
    print(f"    Created {len(ENGINEERED_FEATURES)} new features")

    # 3. Prepare data
    print("\n[3] Preparing data...")
    df = prepare_data(df, ALL_FEATURES, FLOW_PROXY_COL)
    print(f"    Final dataset: {len(df)} rows, {len(ALL_FEATURES)} features")

    # 4. Train-test split (time-based, no shuffle)
    print("\n[4] Splitting data...")
    X = df[ALL_FEATURES]
    y = df[FLOW_PROXY_COL]

    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"    Train: {len(X_train)} samples")
    print(f"    Test:  {len(X_test)} samples")

    # 5. Train regression model
    print("\n[5] Training regression model...")
    reg_model = train_regression_model(X_train_scaled, y_train)

    # Predict
    y_pred_reg = reg_model.predict(X_test_scaled)
    reg_metrics = evaluate_regression(y_test.values, y_pred_reg)

    # Visualizations
    plot_feature_importance(reg_model, ALL_FEATURES, "Regression Feature Importance")
    plot_predictions(y_test.values, y_pred_reg, "Regression: Predicted vs Actual")

    # 6. Train classification model
    print("\n[6] Training classification model...")
    # Create classification labels
    y_train_clf = np.where(y_train > 0, 1, np.where(y_train < 0, -1, 0))
    y_test_clf = np.where(y_test > 0, 1, np.where(y_test < 0, -1, 0))

    clf_model, label_map, reverse_map = train_classification_model(X_train_scaled, y_train_clf)

    # Predict
    y_pred_clf = clf_model.predict(X_test_scaled)
    clf_metrics = evaluate_classification(y_test_clf, y_pred_clf, label_map)

    # 7. Save results
    print("\n[7] Saving results...")
    results_df = pd.DataFrame({
        "actual": y_test.values,
        "predicted_reg": y_pred_reg,
        "predicted_clf": y_pred_clf
    })
    results_df.to_csv(f"{OUTPUT_DIR}/prediction_results.csv", index=False)

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print(f"Results saved to: {OUTPUT_DIR}/")
    print("=" * 60)

    return reg_model, clf_model, scaler


if __name__ == "__main__":
    main()
